In [1]:
!pip install torch joblib matplotlib seaborn tqdm pgmpy fairlearn optuna

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 834.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 41.0 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3


In [2]:
# %% [markdown]
# # Zero-Fairness Pipeline  —  floor(EO)=0.00, floor(DP)=0.00, acc drop < 5%
# Datasets: Adult, COMPAS, German, Bank, Law School, Hospital
# Architecture: BBN Prior → Adversarial Encoder → Joint Threshold Sweep → Optuna HPO

# %% ── CELL 1: Imports & constants ────────────────────────────────────────────
import os, gc, copy, math, time, json, warnings, logging
from dataclasses import dataclass, field, fields as dc_fields
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import accuracy_score
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.isotonic import IsotonicRegression

import joblib
from tqdm import tqdm

from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import BayesianEstimator
from pgmpy.inference import VariableElimination

from fairlearn.metrics import demographic_parity_difference, equalized_odds_difference

try:
    import optuna
    from optuna.samplers import TPESampler
    from optuna.pruners import MedianPruner
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    OPTUNA_AVAILABLE = True
except ImportError:
    OPTUNA_AVAILABLE = False

warnings.filterwarnings('ignore')
logging.getLogger("pgmpy").setLevel(logging.ERROR)

# ── Global constants ──────────────────────────────────────────────────────────
SEED        = 42
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CACHE_DIR   = './cache'
RESULTS_DIR = '/kaggle/working'
BBN_MAX_ROWS     = 2_000
MAX_SWEEP_PAIRS  = 200_000

for _d in [CACHE_DIR, RESULTS_DIR]:
    os.makedirs(_d, exist_ok=True)

def floor2(x: float) -> float:
    return math.floor(abs(float(x)) * 100) / 100

def set_seed(s: int = SEED):
    torch.manual_seed(s)
    np.random.seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

set_seed()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

print(f"Device: {DEVICE}  |  CUDA: {torch.cuda.is_available()}")
print(f"Optuna: {OPTUNA_AVAILABLE}")
# %% ── CELL 2: Hyperparameter dataclass & per-dataset defaults ────────────────

@dataclass
class DatasetHParams:
    # ── Architecture ──────────────────────────────────────────────────────────
    hidden_dim:           int   = 256
    fairness_dim:         int   = 128
    dropout:              float = 0.15
    # ── Optimiser ─────────────────────────────────────────────────────────────
    lr:                   float = 1e-3
    batch_size:           int   = 64
    # ── Stage 1: encoder ──────────────────────────────────────────────────────
    encoder_epochs:       int   = 200
    early_patience:       int   = 20
    bbn_prior_weight:     float = 0.20   # weight of BBN prior loss in stage 1
    # ── Stage 2: adversarial ──────────────────────────────────────────────────
    fairness_epochs:      int   = 300
    encoder_lr_factor:    float = 0.50   # enc_lr = lr * factor
    adversary_lr_factor:  float = 1.00   # adv_lr = lr * factor
    lambda_adv_start:     float = 0.30   # starting adversary weight
    lambda_adv_max:       float = 3.00   # max adversary weight (linearly warmed)
    lambda_warmup_epochs: int   = 20
    cls_loss_weight_s2:   float = 0.50   # classification loss weight in stage 2
    # ── Accuracy budget ───────────────────────────────────────────────────────
    max_acc_drop:         float = 0.050
    stage2_max_acc_drop:  float = 0.055
    acc_drop_fallback:    float = 0.070
    # ── BBN ───────────────────────────────────────────────────────────────────
    use_bbn:              bool  = True
    bbn_weight:           float = 0.30
    bbn_threshold:        float = 0.28
    bbn_fairness_dir:     bool  = True
    # ── Threshold sweep ───────────────────────────────────────────────────────
    use_isotonic:         bool  = True   # apply isotonic calibration before sweep
    use_group_threshold:  bool  = True   # per-group threshold search
    group_thresh_eps:     float = 0.75   # how far group thresholds can deviate
    fine_w:               float = 0.35   # acc weight in threshold objective
    tau:                  float = 0.55   # global threshold starting point


# ── Per-dataset defaults (hand-tuned, HPO refines further) ───────────────────
DEFAULT_HPARAMS: Dict[str, DatasetHParams] = {

    "adult": DatasetHParams(
        hidden_dim=256, fairness_dim=128, dropout=0.15,
        lr=8e-4, batch_size=256,
        encoder_epochs=200, early_patience=25, bbn_prior_weight=0.20,
        fairness_epochs=300, encoder_lr_factor=0.40, adversary_lr_factor=1.40,
        lambda_adv_start=0.25, lambda_adv_max=3.0, lambda_warmup_epochs=12,
        cls_loss_weight_s2=0.50,
        max_acc_drop=0.050, stage2_max_acc_drop=0.060, acc_drop_fallback=0.070,
        use_bbn=True, bbn_weight=0.30, bbn_threshold=0.28, bbn_fairness_dir=True,
        use_isotonic=True, use_group_threshold=True,
        group_thresh_eps=0.75, fine_w=0.35, tau=0.55,
    ),

    "compas": DatasetHParams(
        hidden_dim=256, fairness_dim=128, dropout=0.15,
        lr=1e-3, batch_size=64,
        encoder_epochs=300, early_patience=25, bbn_prior_weight=0.25,
        fairness_epochs=500, encoder_lr_factor=0.60, adversary_lr_factor=1.50,
        lambda_adv_start=0.50, lambda_adv_max=6.0, lambda_warmup_epochs=10,
        cls_loss_weight_s2=0.30,
        max_acc_drop=0.050, stage2_max_acc_drop=0.060, acc_drop_fallback=0.070,
        use_bbn=True, bbn_weight=0.40, bbn_threshold=0.35, bbn_fairness_dir=True,
        use_isotonic=True, use_group_threshold=True,
        group_thresh_eps=0.99, fine_w=0.45, tau=0.65,
    ),

    "german": DatasetHParams(
        hidden_dim=192, fairness_dim=96, dropout=0.12,
        lr=5e-4, batch_size=32,
        encoder_epochs=400, early_patience=25, bbn_prior_weight=0.20,
        fairness_epochs=500, encoder_lr_factor=0.60, adversary_lr_factor=1.20,
        lambda_adv_start=0.30, lambda_adv_max=4.0, lambda_warmup_epochs=10,
        cls_loss_weight_s2=0.40,
        max_acc_drop=0.050, stage2_max_acc_drop=0.060, acc_drop_fallback=0.070,
        use_bbn=True, bbn_weight=0.35, bbn_threshold=0.45, bbn_fairness_dir=True,
        use_isotonic=False, use_group_threshold=True,
        group_thresh_eps=0.99, fine_w=0.45, tau=0.40,
    ),

    "bank": DatasetHParams(
        hidden_dim=224, fairness_dim=112, dropout=0.10,
        lr=1e-3, batch_size=256,
        encoder_epochs=70, early_patience=20, bbn_prior_weight=0.10,
        fairness_epochs=100, encoder_lr_factor=0.0, adversary_lr_factor=1.50,
        lambda_adv_start=0.05, lambda_adv_max=0.40, lambda_warmup_epochs=10,
        cls_loss_weight_s2=0.60,
        max_acc_drop=0.050, stage2_max_acc_drop=0.055, acc_drop_fallback=0.070,
        use_bbn=True, bbn_weight=0.20, bbn_threshold=0.18, bbn_fairness_dir=False,
        use_isotonic=False, use_group_threshold=True,
        group_thresh_eps=0.10, fine_w=0.12, tau=0.50,
    ),

    "lawschool": DatasetHParams(
        hidden_dim=192, fairness_dim=96, dropout=0.12,
        lr=5e-4, batch_size=256,
        encoder_epochs=120, early_patience=20, bbn_prior_weight=0.15,
        fairness_epochs=120, encoder_lr_factor=0.0, adversary_lr_factor=1.50,
        lambda_adv_start=0.08, lambda_adv_max=0.50, lambda_warmup_epochs=15,
        cls_loss_weight_s2=0.85,
        max_acc_drop=0.050, stage2_max_acc_drop=0.055, acc_drop_fallback=0.070,
        use_bbn=True, bbn_weight=0.25, bbn_threshold=0.18, bbn_fairness_dir=True,
        use_isotonic=False, use_group_threshold=True,
        group_thresh_eps=0.12, fine_w=0.15, tau=0.45,
    ),

    "hospital": DatasetHParams(
        hidden_dim=288, fairness_dim=144, dropout=0.20,
        lr=9e-4, batch_size=128,
        encoder_epochs=150, early_patience=25, bbn_prior_weight=0.20,
        fairness_epochs=200, encoder_lr_factor=0.15, adversary_lr_factor=1.0,
        lambda_adv_start=0.15, lambda_adv_max=1.0, lambda_warmup_epochs=15,
        cls_loss_weight_s2=0.50,
        max_acc_drop=0.050, stage2_max_acc_drop=0.055, acc_drop_fallback=0.070,
        use_bbn=True, bbn_weight=0.22, bbn_threshold=0.18, bbn_fairness_dir=False,
        use_isotonic=False, use_group_threshold=True,
        group_thresh_eps=0.15, fine_w=0.20, tau=0.50,
    ),
}

# Sensitive attribute key mapping (used for evaluate_agent & HPO)
DATASET_CONFIG: Dict[str, Dict] = {
    "adult":     {"sens_attrs": [("sens_sex_train",     "sens_sex_test"),
                                 ("sens_race_train",    "sens_race_test")]},
    "compas":    {"sens_attrs": [("sens_race_train",    "sens_race_test"),
                                 ("sens_sex_train",     "sens_sex_test")]},
    "german":    {"sens_attrs": [("sens_age_train",     "sens_age_test"),
                                 ("sens_sex_train",     "sens_sex_test")]},
    "bank":      {"sens_attrs": [("sens_marital_train", "sens_marital_test"),
                                 ("sens_job_train",     "sens_job_test")]},
    "lawschool": {"sens_attrs": [("sens_race_train",    "sens_race_test"),
                                 ("sens_gender_train",  "sens_gender_test")]},
    "hospital":  {"sens_attrs": [("sens_race_train",    "sens_race_test"),
                                 ("sens_sex_train",     "sens_sex_test")]},
}

print("DatasetHParams and DEFAULT_HPARAMS defined.")
print(f"Datasets: {list(DEFAULT_HPARAMS.keys())}")
# %% ── CELL 3: Utility helpers & fairness metrics ────────────────────────────

def to_dense(X) -> np.ndarray:
    arr = X.toarray() if hasattr(X, "toarray") else np.asarray(X)
    return np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)

def clean_numeric_column(s: pd.Series) -> pd.Series:
    s = pd.to_numeric(s, errors='coerce').replace([np.inf, -np.inf], np.nan)
    med = s.median()
    return s.fillna(0.0 if np.isnan(med) else med)

def safe_qcut(s: pd.Series, q: int = 5) -> pd.Series:
    s = clean_numeric_column(s)
    if s.nunique() <= 1:
        return pd.Series(0, index=s.index, dtype=int)
    try:
        return pd.qcut(s, q, labels=False, duplicates='drop').fillna(0).astype(int)
    except Exception:
        try:
            return pd.cut(s, q, labels=False, duplicates='drop').fillna(0).astype(int)
        except Exception:
            return pd.Series(0, index=s.index, dtype=int)

def make_num_pipeline():
    return Pipeline([('imp', SimpleImputer(strategy='median')),
                     ('sc',  StandardScaler())])

def make_cat_pipeline():
    return Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                     ('ohe', OneHotEncoder(handle_unknown='ignore'))])

# ── Fairness metrics ──────────────────────────────────────────────────────────

def compute_eo(y_true, y_pred, s):
    """Max of |ΔTPR| and |ΔFPR| across the two groups."""
    g = np.unique(s)
    if len(g) != 2:
        return 0.0
    tpr, fpr = [], []
    for gi in g:
        pos = (s == gi) & (y_true == 1)
        neg = (s == gi) & (y_true == 0)
        tpr.append(y_pred[pos].mean() if pos.sum() > 0 else 0.0)
        fpr.append(y_pred[neg].mean() if neg.sum() > 0 else 0.0)
    return float(max(abs(tpr[0] - tpr[1]), abs(fpr[0] - fpr[1])))

def compute_dp(y_pred, s):
    """|positive rate group0 - positive rate group1|."""
    g = np.unique(s)
    if len(g) != 2:
        return 0.0
    rates = [y_pred[s == gi].mean() for gi in g]
    return float(abs(rates[0] - rates[1]))

def compute_fairness_metrics(y_true, y_pred, sensitive_dict: dict) -> dict:
    """
    Returns {attr_eo: float, attr_dp: float, ...} for every sensitive attribute.
    Uses fairlearn when available, falls back to custom implementation.
    """
    metrics = {}
    yt = np.asarray(y_true)
    yp = np.asarray(y_pred)
    for name, values in sensitive_dict.items():
        sv = np.asarray(values).astype(int).flatten()
        if np.unique(sv).size != 2:
            metrics[f"{name}_eo"] = 0.0
            metrics[f"{name}_dp"] = 0.0
            continue
        try:
            eo = equalized_odds_difference(yt, yp, sensitive_features=sv)
            metrics[f"{name}_eo"] = 0.0 if np.isnan(eo) else abs(float(eo))
        except Exception:
            metrics[f"{name}_eo"] = compute_eo(yt, yp, sv)
        try:
            dp = demographic_parity_difference(yt, yp, sensitive_features=sv)
            metrics[f"{name}_dp"] = 0.0 if np.isnan(dp) else abs(float(dp))
        except Exception:
            metrics[f"{name}_dp"] = compute_dp(yp, sv)
    return metrics

def _sens_list_from_data(data: dict, dataset_key: str) -> List[np.ndarray]:
    """Return list of test sensitive attribute arrays for a dataset."""
    cfg = DATASET_CONFIG[dataset_key]
    out = []
    for _, test_key in cfg["sens_attrs"]:
        if test_key in data:
            out.append(np.asarray(data[test_key]).flatten())
    return out

def _sens_dict_from_data(data: dict, dataset_key: str) -> dict:
    cfg = DATASET_CONFIG[dataset_key]
    out = {}
    for _, test_key in cfg["sens_attrs"]:
        if test_key in data:
            attr_name = test_key.replace("sens_", "").replace("_test", "")
            out[attr_name] = np.asarray(data[test_key]).flatten()
    return out

def _sens_train_list(data: dict, dataset_key: str) -> List[np.ndarray]:
    cfg = DATASET_CONFIG[dataset_key]
    out = []
    for train_key, _ in cfg["sens_attrs"]:
        if train_key in data:
            out.append(np.asarray(data[train_key]).flatten())
    return out

print("Utility helpers and fairness metrics defined.")
# %% ── CELL 4: Preprocessing functions (all 6 datasets) ─────────────────────

def preprocess_adult_for_fair_bbn(
        csv_path='/kaggle/input/datasets/maansirao/all-datasets/adult.csv',
        seed=SEED, use_cache=True):
    cache_file = os.path.join(CACHE_DIR, 'preproc_adult.pkl')
    if use_cache and os.path.exists(cache_file):
        tqdm.write("  [cache] Adult data loaded.")
        return joblib.load(cache_file)

    col_names = ['age','workclass','fnlwgt','education','education-num',
                 'marital-status','occupation','relationship','race','sex',
                 'capital-gain','capital-loss','hours-per-week','native-country','income']
    df = None
    for sep in [', ', ',']:
        try:
            peek = pd.read_csv(csv_path, sep=sep, nrows=1, header=0)
            if peek.shape[1] == 15:
                first_val = str(peek.iloc[0, 0]).strip()
                if first_val.lstrip('-').isdigit():
                    df = pd.read_csv(csv_path, sep=sep, names=col_names, header=None)
                else:
                    df = pd.read_csv(csv_path, sep=sep, header=0)
                    df.columns = col_names
                break
        except Exception:
            continue
    if df is None:
        df = pd.read_csv(csv_path)
        if df.shape[1] == 15:
            df.columns = col_names
        else:
            raise ValueError(f"Cannot parse Adult CSV: shape {df.shape}")
    df.columns = col_names

    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype(str).str.strip()
    df = df[~df.isin(['?']).any(axis=1)].reset_index(drop=True)
    df = df.drop(columns=['fnlwgt'])

    income_col = df['income'].astype(str).str.strip()
    df['income_label'] = income_col.str.contains('>50K', na=False).astype(int)
    if df['income_label'].sum() == 0:
        df['income_label'] = pd.to_numeric(df['income'], errors='coerce').fillna(0).astype(int)

    df['sex_binary']  = (df['sex'].astype(str).str.strip() == 'Male').astype(int)
    df['race_binary'] = (df['race'].astype(str).str.strip() == 'White').astype(int)

    numeric_cols    = ['age','education-num','capital-gain','capital-loss','hours-per-week']
    categorical_cols = ['workclass','education','marital-status','occupation',
                        'relationship','native-country']
    for col in numeric_cols:
        df[col] = clean_numeric_column(df[col])
    for col in ['capital-gain','capital-loss']:
        df[col] = df[col].clip(upper=df[col].quantile(0.99))

    preproc = ColumnTransformer([('num', make_num_pipeline(), numeric_cols),
                                 ('cat', make_cat_pipeline(), categorical_cols)])

    bbn_df = df.copy()
    for col in numeric_cols:
        bbn_df[col] = safe_qcut(bbn_df[col], 5)
    for col in categorical_cols + ['sex', 'race']:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)

    X = df.drop(columns=['income','income_label','sex_binary','race_binary','sex','race'])
    y = df['income_label'].values
    for col in X.select_dtypes(include=[np.number]).columns:
        X[col] = clean_numeric_column(X[col])

    (X_tr, X_te, y_tr, y_te, ss_tr, ss_te, sr_tr, sr_te) = train_test_split(
        X, y, df['sex_binary'].values, df['race_binary'].values,
        test_size=0.2, stratify=y, random_state=seed)

    X_train_nn = to_dense(preproc.fit_transform(X_tr))
    X_test_nn  = to_dense(preproc.transform(X_te))
    bbn_train  = bbn_df.loc[X_tr.index].reset_index(drop=True)
    bbn_test   = bbn_df.loc[X_te.index].reset_index(drop=True)

    result = dict(X_train_nn=X_train_nn, X_test_nn=X_test_nn,
                  bbn_train_df=bbn_train, bbn_test_df=bbn_test,
                  y_train=y_tr, y_test=y_te,
                  sens_sex_train=ss_tr, sens_sex_test=ss_te,
                  sens_race_train=sr_tr, sens_race_test=sr_te)
    if use_cache: joblib.dump(result, cache_file)
    return result


def preprocess_compas_for_fair_bbn(
        csv_path='/kaggle/input/datasets/maansirao/all-datasets/compas-scores-two-years.csv',
        seed=SEED, use_cache=True):
    cache_file = os.path.join(CACHE_DIR, 'preproc_compas.pkl')
    if use_cache and os.path.exists(cache_file):
        tqdm.write("  [cache] COMPAS data loaded.")
        return joblib.load(cache_file)

    df = pd.read_csv(csv_path)
    df = df.loc[
        (df['days_b_screening_arrest'] <= 30) & (df['days_b_screening_arrest'] >= -30) &
        (df['is_recid'] != -1) & (df['c_charge_degree'] != 'O') & (df['score_text'] != 'N/A'),
        ['age','c_charge_degree','race','age_cat','score_text','sex','priors_count',
         'days_b_screening_arrest','decile_score','juv_other_count','juv_misd_count',
         'juv_fel_count','c_charge_desc','is_recid','two_year_recid','c_jail_in','c_jail_out']
    ].reset_index(drop=True)

    df['race_label'] = df['race'].map({"African-American":0,"Caucasian":1,"Hispanic":2,
                                        "Other":3,"Asian":4,"Native American":5})
    df['sex_label']  = df['sex'].map({"Male":0,"Female":1})
    df['c_jail_in']  = pd.to_datetime(df['c_jail_in'])
    df['c_jail_out'] = pd.to_datetime(df['c_jail_out'])
    df['jail_time']  = (df['c_jail_out'] - df['c_jail_in']).dt.days.fillna(0)
    df = df.drop(columns=['c_jail_in','c_jail_out'])
    df['race_binary'] = (df['race_label'] == 0).astype(int)
    df['sex_binary']  = df['sex_label']

    numeric_cols    = ['age','priors_count','days_b_screening_arrest','decile_score',
                       'jail_time','juv_other_count','juv_misd_count','juv_fel_count']
    categorical_cols = ['c_charge_degree','race','age_cat','score_text','sex','c_charge_desc']
    for col in numeric_cols:
        df[col] = clean_numeric_column(df[col])

    preproc = ColumnTransformer([('num', make_num_pipeline(), numeric_cols),
                                 ('cat', make_cat_pipeline(), categorical_cols)])

    bbn_df = df.copy()
    for col in numeric_cols:
        bbn_df[col] = safe_qcut(bbn_df[col], 5)
    for col in categorical_cols + ['race','sex']:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)

    X = df.drop(columns=['is_recid','two_year_recid','race_label','sex_label',
                          'race_binary','sex_binary'])
    y = df['two_year_recid'].values
    (X_tr, X_te, y_tr, y_te, sr_tr, sr_te, ss_tr, ss_te) = train_test_split(
        X, y, df['race_binary'], df['sex_binary'],
        test_size=0.2, stratify=y, random_state=seed)

    X_train_nn = to_dense(preproc.fit_transform(X_tr))
    X_test_nn  = to_dense(preproc.transform(X_te))
    bbn_train  = bbn_df.loc[X_tr.index].reset_index(drop=True)
    bbn_test   = bbn_df.loc[X_te.index].reset_index(drop=True)

    result = dict(X_train_nn=X_train_nn, X_test_nn=X_test_nn,
                  bbn_train_df=bbn_train, bbn_test_df=bbn_test,
                  y_train=y_tr, y_test=y_te,
                  sens_race_train=sr_tr.reset_index(drop=True),
                  sens_race_test=sr_te.reset_index(drop=True),
                  sens_sex_train=ss_tr.reset_index(drop=True),
                  sens_sex_test=ss_te.reset_index(drop=True))
    if use_cache: joblib.dump(result, cache_file)
    return result


def preprocess_german_for_fair_bbn(
        csv_path="/kaggle/input/datasets/maansirao/all-datasets/german.data",
        seed=SEED, use_cache=True):
    cache_file = os.path.join(CACHE_DIR, 'preproc_german.pkl')
    if use_cache and os.path.exists(cache_file):
        tqdm.write("  [cache] German data loaded.")
        return joblib.load(cache_file)

    col_names = ["status","duration","credit_history","purpose","amount","savings",
                 "employment","installment_rate","personal_status_sex","other_debtors",
                 "residence","property","age","other_installment_plans","housing",
                 "number_credits","job","people_liable","telephone","foreign_worker","credit"]
    df = pd.read_csv(csv_path, sep=' ', names=col_names)
    sex_map = {'A91':'male','A92':'male','A93':'male','A94':'female','A95':'female'}
    df['sex'] = df['personal_status_sex'].map(sex_map)
    df['sex_label']            = df['sex'].map({'male':0,'female':1})
    df['age_label']            = (df['age'] >= 25).astype(int)
    df['foreign_worker_label'] = df['foreign_worker'].map({'A201':1,'A202':0})
    df['credit_label']         = df['credit'].map({1:1, 2:0})
    df = df.drop(columns=['personal_status_sex','sex','age','foreign_worker','credit'])

    numerical_cols  = ["duration","amount","installment_rate","residence",
                       "number_credits","people_liable"]
    categorical_cols = [c for c in df.columns if c not in
                        numerical_cols + ['sex_label','age_label','foreign_worker_label','credit_label']]
    for col in numerical_cols:
        df[col] = clean_numeric_column(df[col])

    preproc = ColumnTransformer([('num', make_num_pipeline(), numerical_cols),
                                 ('cat', make_cat_pipeline(), categorical_cols)])

    bbn_df = df.copy()
    for col in numerical_cols:
        bbn_df[col] = safe_qcut(bbn_df[col], 5)
    for col in categorical_cols + ['sex_label','age_label','foreign_worker_label']:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)

    X = df.drop(columns=['credit_label'])
    y = df['credit_label'].values
    (X_tr, X_te, y_tr, y_te, sa_tr, sa_te, ss_tr, ss_te) = train_test_split(
        X, y, df['age_label'].values, df['sex_label'].values,
        test_size=0.2, stratify=y, random_state=seed)

    X_train_nn = to_dense(preproc.fit_transform(X_tr))
    X_test_nn  = to_dense(preproc.transform(X_te))
    bbn_train  = bbn_df.loc[X_tr.index].reset_index(drop=True)
    bbn_test   = bbn_df.loc[X_te.index].reset_index(drop=True)

    result = dict(X_train_nn=X_train_nn, X_test_nn=X_test_nn,
                  bbn_train_df=bbn_train, bbn_test_df=bbn_test,
                  y_train=y_tr, y_test=y_te,
                  sens_age_train=sa_tr, sens_age_test=sa_te,
                  sens_sex_train=ss_tr, sens_sex_test=ss_te)
    if use_cache: joblib.dump(result, cache_file)
    return result


def preprocess_bank_for_fair_bbn(
        csv_path='/kaggle/input/datasets/maansirao/all-datasets/bank-full.csv',
        seed=SEED, use_cache=True):
    cache_file = os.path.join(CACHE_DIR, 'preproc_bank.pkl')
    if use_cache and os.path.exists(cache_file):
        tqdm.write("  [cache] Bank data loaded.")
        return joblib.load(cache_file)

    try:
        df = pd.read_csv(csv_path, sep=';')
    except Exception:
        df = pd.read_csv(csv_path, sep=',')
    df = df.apply(lambda x: x.str.lower() if x.dtype == 'object' else x)
    target_col = ('y' if 'y' in df.columns else 'deposit' if 'deposit' in df.columns else df.columns[-1])
    df = df[~df.isin(['unknown']).any(axis=1)].reset_index(drop=True)
    df['y'] = np.where(df[target_col].isin(['yes','y','true','1']), 1, 0)
    df['marital_binary'] = (df['marital'] == df['marital'].value_counts().idxmax()).astype(int)
    df['job_binary']     = (df['job']     == df['job'].value_counts().idxmax()).astype(int)

    categorical_cols = [c for c in ['job','education','default','housing','loan',
                                    'contact','month','poutcome'] if c in df.columns]
    numeric_cols     = [c for c in ['age','balance','day','duration','campaign','pdays','previous']
                        if c in df.columns]
    for col in numeric_cols:
        df[col] = clean_numeric_column(df[col])
    for col in ['balance','duration','pdays','previous']:
        if col in df.columns:
            df[col] = df[col].clip(upper=df[col].quantile(0.99))

    preproc = ColumnTransformer([('num', make_num_pipeline(), numeric_cols),
                                 ('cat', make_cat_pipeline(), categorical_cols)])

    bbn_df = df.copy()
    for col in numeric_cols:
        bbn_df[col] = safe_qcut(bbn_df[col], 5)
    for col in categorical_cols + ['marital','job']:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)

    X = df.drop(columns=['y','marital_binary','job_binary'])
    y = df['y'].values
    (X_tr, X_te, y_tr, y_te, sm_tr, sm_te, sj_tr, sj_te) = train_test_split(
        X, y, df['marital_binary'], df['job_binary'],
        test_size=0.2, stratify=y, random_state=seed)

    X_train_nn = to_dense(preproc.fit_transform(X_tr))
    X_test_nn  = to_dense(preproc.transform(X_te))
    bbn_train  = bbn_df.loc[X_tr.index].reset_index(drop=True)
    bbn_test   = bbn_df.loc[X_te.index].reset_index(drop=True)

    result = dict(X_train_nn=X_train_nn, X_test_nn=X_test_nn,
                  bbn_train_df=bbn_train, bbn_test_df=bbn_test,
                  y_train=y_tr, y_test=y_te,
                  sens_marital_train=sm_tr.reset_index(drop=True),
                  sens_marital_test=sm_te.reset_index(drop=True),
                  sens_job_train=sj_tr.reset_index(drop=True),
                  sens_job_test=sj_te.reset_index(drop=True))
    if use_cache: joblib.dump(result, cache_file)
    return result


def preprocess_lawschool_for_fair_bbn(
        law_path="/kaggle/input/datasets/maansirao/all-datasets/bar_pass_prediction.csv",
        use_cache=True):
    cache_file = os.path.join(CACHE_DIR, 'preproc_lawschool.pkl')
    if use_cache and os.path.exists(cache_file):
        tqdm.write("  [cache] Law School data loaded.")
        return joblib.load(cache_file)

    law_df = pd.read_csv(law_path)
    law_df.columns = [c.strip().lower() for c in law_df.columns]
    law_df = law_df.dropna(subset=['pass_bar','race','sex'])
    for col in law_df.select_dtypes(include=[np.number]).columns:
        law_df[col] = clean_numeric_column(law_df[col])

    most_common_race      = law_df['race'].value_counts().idxmax()
    law_df['race_binary'] = (law_df['race'] == most_common_race).astype(int)
    gender_map            = {v: i for i, v in enumerate(sorted(law_df['sex'].unique()))}
    law_df['gender_binary'] = law_df['sex'].map(gender_map)

    numeric_cols    = [c for c in ['lsat','ugpa','zfygpa','zgpa','fam_inc','age','gpa']
                       if c in law_df.columns and law_df[c].nunique() > 1]
    categorical_cols = [c for c in ['tier','cluster','fulltime','parttime']
                        if c in law_df.columns]

    preproc = ColumnTransformer([('num', make_num_pipeline(), numeric_cols),
                                 ('cat', make_cat_pipeline(), categorical_cols)])

    bbn_df = law_df.copy()
    for col in numeric_cols:
        bbn_df[col] = safe_qcut(law_df[col], 5)
    for col in categorical_cols + ['race','sex']:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)

    X = law_df[numeric_cols + categorical_cols + ['race','sex']]
    y = law_df['pass_bar'].values
    (X_tr, X_te, y_tr, y_te, sr_tr, sr_te, sg_tr, sg_te) = train_test_split(
        X, y, law_df['race_binary'], law_df['gender_binary'],
        test_size=0.2, stratify=y, random_state=SEED)

    X_train_nn = to_dense(preproc.fit_transform(X_tr))
    X_test_nn  = to_dense(preproc.transform(X_te))
    bbn_train  = bbn_df.loc[X_tr.index].reset_index(drop=True)
    bbn_test   = bbn_df.loc[X_te.index].reset_index(drop=True)

    result = dict(X_train_nn=X_train_nn, X_test_nn=X_test_nn,
                  bbn_train_df=bbn_train, bbn_test_df=bbn_test,
                  y_train=y_tr, y_test=y_te,
                  sens_race_train=sr_tr.reset_index(drop=True),
                  sens_race_test=sr_te.reset_index(drop=True),
                  sens_gender_train=sg_tr.reset_index(drop=True),
                  sens_gender_test=sg_te.reset_index(drop=True))
    if use_cache: joblib.dump(result, cache_file)
    return result


def preprocess_diabetes_hospital_for_fair_bbn(
        csv_path='/kaggle/input/datasets/maansirao/all-datasets/diabetes_hospital_fairlearn.csv',
        seed=SEED, use_cache=True):
    cache_file = os.path.join(CACHE_DIR, 'preproc_hospital.pkl')
    if use_cache and os.path.exists(cache_file):
        tqdm.write("  [cache] Hospital data loaded.")
        return joblib.load(cache_file)

    df = pd.read_csv(csv_path)
    df = df.drop(columns=[c for c in ["max_glu_serum","A1Cresult","readmitted.1"]
                           if c in df.columns])
    df = df[~df.isin(['Missing']).any(axis=1)]
    df = df.dropna(subset=['race','gender']).reset_index(drop=True)

    age_map = {"'0-10'":5,"'10-20'":15,"'20-30'":25,"'30-40'":35,"'40-50'":45,
               "'50-60'":55,"'60-70'":65,"'70-80'":75,"'80-90'":85,"'90-100'":95,
               "'30 years or younger'":20,"'30-60 years'":45,"'Over 60 years'":70}
    df['age'] = df['age'].replace(age_map).astype(float)
    df['readmit_binary'] = df['readmit_binary'].astype(int)

    most_common_race    = df['race'].value_counts().idxmax()
    df['race_binary']   = (df['race'] == most_common_race).astype(int)
    df['gender_binary'] = df['gender'].map({'Male':0,'Female':1}).fillna(0).astype(int)

    categorical_cols = ['discharge_disposition_id','admission_source_id','medical_specialty',
                        'primary_diagnosis','insulin','change','diabetesMed']
    numeric_cols     = ['age','time_in_hospital','num_lab_procedures','num_procedures',
                        'num_medications','number_diagnoses','had_emergency',
                        'had_inpatient_days','had_outpatient_days','medicare','medicaid']
    for col in numeric_cols:
        df[col] = clean_numeric_column(df[col])

    preproc = ColumnTransformer([('num', make_num_pipeline(), numeric_cols),
                                 ('cat', make_cat_pipeline(), categorical_cols)])

    bbn_df = df.copy()
    for col in numeric_cols:
        bbn_df[col] = safe_qcut(bbn_df[col], 5)
    for col in categorical_cols + ['race','gender']:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)

    X = df.drop(columns=['readmit_binary','race_binary','gender_binary'])
    y = df['readmit_binary'].values
    (X_tr, X_te, y_tr, y_te, sr_tr, sr_te, sg_tr, sg_te) = train_test_split(
        X, y, df['race_binary'], df['gender_binary'],
        test_size=0.2, stratify=y, random_state=seed)

    X_train_nn = to_dense(preproc.fit_transform(X_tr))
    X_test_nn  = to_dense(preproc.transform(X_te))
    bbn_train  = bbn_df.loc[X_tr.index].reset_index(drop=True)
    bbn_test   = bbn_df.loc[X_te.index].reset_index(drop=True)

    result = dict(X_train_nn=X_train_nn, X_test_nn=X_test_nn,
                  bbn_train_df=bbn_train, bbn_test_df=bbn_test,
                  y_train=y_tr, y_test=y_te,
                  sens_race_train=sr_tr.reset_index(drop=True),
                  sens_race_test=sr_te.reset_index(drop=True),
                  sens_sex_train=sg_tr.reset_index(drop=True),
                  sens_sex_test=sg_te.reset_index(drop=True))
    if use_cache: joblib.dump(result, cache_file)
    return result

print("All 6 preprocessing functions defined.")
# %% ── CELL 5: BBN model & neural network architecture ───────────────────────

class BBNFairnessModel:
    def __init__(self, target_col='label', sens_cols=None, max_rows=BBN_MAX_ROWS):
        self.target_col = target_col
        self.sens_cols  = sens_cols or []
        self.max_rows   = max_rows
        self.model      = None
        self.infer      = None
        self.fitted     = False

    def fit(self, bbn_df: pd.DataFrame, y_train: np.ndarray):
        df = bbn_df.copy()
        df[self.target_col] = y_train.astype(int)
        if len(df) > self.max_rows:
            df = df.sample(self.max_rows, random_state=SEED)
        for col in df.columns:
            df[col] = df[col].astype(int).clip(0, 9)
        edges = [(col, self.target_col) for col in self.sens_cols if col in df.columns]
        if not edges:
            self.fitted = False
            return self
        try:
            self.model = DiscreteBayesianNetwork(edges)
            self.model.fit(df[[c for _, c in [('', t) for t in
                                [self.target_col] + self.sens_cols] if c in df.columns]
                               + [s for s in self.sens_cols if s in df.columns]
                               + [self.target_col]][:1],
                           estimator=BayesianEstimator, prior_type='BDeu', equivalent_sample_size=5)
            cols_needed = list({s for s in self.sens_cols if s in df.columns} | {self.target_col})
            self.model = DiscreteBayesianNetwork(edges)
            self.model.fit(df[cols_needed], estimator=BayesianEstimator,
                           prior_type='BDeu', equivalent_sample_size=5)
            self.infer  = VariableElimination(self.model)
            self.fitted = True
        except Exception as e:
            self.fitted = False
        return self

    def predict_proba(self, bbn_df: pd.DataFrame) -> np.ndarray:
        if not self.fitted or self.infer is None:
            return np.full(len(bbn_df), 0.5)
        probs = np.full(len(bbn_df), 0.5)
        for i, row in bbn_df.iterrows():
            try:
                evidence = {col: int(row[col]) for col in self.sens_cols
                            if col in bbn_df.columns and not pd.isna(row[col])}
                if not evidence:
                    continue
                q = self.infer.query([self.target_col], evidence=evidence, show_progress=False)
                vals = q.values
                probs[i] = float(vals[1]) if len(vals) > 1 else float(vals[0])
            except Exception:
                pass
        return probs


class FairnessEncoder(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, fairness_dim: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, fairness_dim),
            nn.BatchNorm1d(fairness_dim),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x)


class Classifier(nn.Module):
    def __init__(self, fairness_dim: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(fairness_dim, fairness_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(fairness_dim // 2, 1),
        )

    def forward(self, z):
        return self.net(z).squeeze(-1)


class SensitivityAdversary(nn.Module):
    def __init__(self, fairness_dim: int, n_sensitive: int, dropout: float):
        super().__init__()
        self.heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(fairness_dim, fairness_dim // 2),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(fairness_dim // 2, 1),
            ) for _ in range(n_sensitive)
        ])

    def forward(self, z):
        return [h(z).squeeze(-1) for h in self.heads]


print("BBN model and neural network modules defined.")
# %% ── CELL 6: Threshold sweep ───────────────────────────────────────────────

def _eo_dp_at_threshold(y_true, y_proba, s_list, threshold):
    y_pred = (y_proba >= threshold).astype(int)
    eo = max((compute_eo(y_true, y_pred, s) for s in s_list), default=0.0)
    dp = max((compute_dp(y_pred, s) for s in s_list), default=0.0)
    return eo, dp

def _group_predictions(y_proba, s_list, thresholds):
    n = len(y_proba)
    y_pred = np.zeros(n, dtype=int)
    for i in range(n):
        t = thresholds[0]
        for j, s in enumerate(s_list):
            if j < len(thresholds):
                t = thresholds[j] if s[i] == 1 else thresholds[j]
        y_pred[i] = int(y_proba[i] >= t)
    return y_pred

def joint_threshold_sweep(
        y_true: np.ndarray,
        y_proba: np.ndarray,
        s_list: List[np.ndarray],
        hp: DatasetHParams,
        baseline_acc: float,
) -> Tuple[np.ndarray, float]:
    """
    Sweep global thresholds to find the one that achieves
    floor(EO)=0.00 AND floor(DP)=0.00 with minimum accuracy loss.
    Falls back gracefully if no single threshold achieves the target.

    Returns (y_pred_binary, threshold_used).
    """
    y_true   = np.asarray(y_true)
    y_proba  = np.asarray(y_proba, dtype=float)
    max_drop = hp.acc_drop_fallback

    thresholds = np.linspace(0.10, 0.90, 161)
    best_pred  = (y_proba >= hp.tau).astype(int)
    best_score = np.inf
    best_tau   = hp.tau

    for tau in thresholds:
        yp  = (y_proba >= tau).astype(int)
        acc = accuracy_score(y_true, yp)
        if acc < baseline_acc - max_drop:
            continue
        eo = max((compute_eo(y_true, yp, s) for s in s_list), default=0.0)
        dp = max((compute_dp(yp, s) for s in s_list), default=0.0)
        penalty = floor2(eo) * 20 + floor2(dp) * 10
        acc_drop = max(0.0, baseline_acc - acc)
        score    = penalty + (acc_drop if penalty == 0.0 else 0.0)
        if score < best_score:
            best_score = score
            best_pred  = yp
            best_tau   = tau

    if not hp.use_group_threshold:
        return best_pred, best_tau

    eo_global = max((compute_eo(y_true, best_pred, s) for s in s_list), default=0.0)
    dp_global = max((compute_dp(best_pred, s) for s in s_list), default=0.0)
    if floor2(eo_global) == 0.0 and floor2(dp_global) == 0.0:
        return best_pred, best_tau

    eps   = hp.group_thresh_eps
    fine  = hp.fine_w
    n_s   = len(s_list)
    n_pts = max(1, int((MAX_SWEEP_PAIRS ** (1/n_s))))
    n_pts = min(n_pts, 25)
    grids = [np.linspace(max(0.05, best_tau - eps * best_tau),
                         min(0.95, best_tau + eps * (1 - best_tau)),
                         n_pts) for _ in range(n_s)]

    from itertools import product as iproduct
    best_gp   = best_pred.copy()
    best_gs   = best_score

    for combo in iproduct(*grids):
        tmap = {j: combo[j] for j in range(n_s)}
        n    = len(y_proba)
        yp   = np.zeros(n, dtype=int)
        for i in range(n):
            t = best_tau
            for j, s in enumerate(s_list):
                if j < n_s:
                    t = tmap[j] if s[i] == 1 else combo[min(j+1, n_s-1)] \
                        if j + 1 < n_s else combo[j]
                    break
            yp[i] = int(y_proba[i] >= t)
        acc = accuracy_score(y_true, yp)
        if acc < baseline_acc - max_drop:
            continue
        eo = max((compute_eo(y_true, yp, s) for s in s_list), default=0.0)
        dp = max((compute_dp(yp, s) for s in s_list), default=0.0)
        penalty  = floor2(eo) * 20 + floor2(dp) * 10
        acc_drop = max(0.0, baseline_acc - acc)
        score    = penalty * (1 - fine) + acc_drop * fine
        if score < best_gs:
            best_gs = score
            best_gp = yp

    return best_gp, best_tau


def calibrate_proba(y_proba_train, y_train, y_proba_test):
    iso = IsotonicRegression(out_of_bounds='clip')
    iso.fit(y_proba_train, y_train)
    return iso.transform(y_proba_test), iso.transform(y_proba_train)


print("Threshold sweep defined.")
# %% ── CELL 7: train_model  (stage 1 → stage 2 → threshold sweep) ────────────

def _tensor(arr, dtype=torch.float32):
    return torch.tensor(np.asarray(arr), dtype=dtype).to(DEVICE)

def _get_proba(encoder, classifier, X_tensor, batch=2048):
    encoder.eval(); classifier.eval()
    parts = []
    with torch.no_grad():
        for i in range(0, len(X_tensor), batch):
            z = encoder(X_tensor[i:i+batch])
            parts.append(torch.sigmoid(classifier(z)).cpu().numpy())
    return np.concatenate(parts)


def train_model(
        data: dict,
        dataset_key: str,
        hp: DatasetHParams,
        original_baseline_acc: float,
        verbose: bool = False,
) -> dict:
    set_seed(SEED)
    y_train = np.asarray(data['y_train'])
    y_test  = np.asarray(data['y_test'])
    X_tr    = _tensor(data['X_train_nn'])
    X_te    = _tensor(data['X_test_nn'])

    s_train = _sens_train_list(data, dataset_key)
    s_test  = _sens_list_from_data(data, dataset_key)
    s_dict  = _sens_dict_from_data(data, dataset_key)
    n_sens  = len(s_train)

    input_dim = X_tr.shape[1]

    encoder    = FairnessEncoder(input_dim, hp.hidden_dim, hp.fairness_dim, hp.dropout).to(DEVICE)
    classifier = Classifier(hp.fairness_dim, hp.dropout).to(DEVICE)
    adversary  = SensitivityAdversary(hp.fairness_dim, n_sens, hp.dropout).to(DEVICE)

    pos_weight = torch.tensor([(y_train == 0).sum() / max(1, (y_train == 1).sum())],
                               dtype=torch.float32).to(DEVICE)
    bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    bce_unw = nn.BCEWithLogitsLoss()

    yt_tr = _tensor(y_train)
    s_tensors_tr = [_tensor(s.astype(float)) for s in s_train]

    ds_tr = TensorDataset(X_tr, yt_tr, *s_tensors_tr)
    dl_tr = DataLoader(ds_tr, batch_size=hp.batch_size, shuffle=True, drop_last=False)

    # ── BBN prior ─────────────────────────────────────────────────────────────
    bbn_prior_train = np.full(len(y_train), 0.5)
    if hp.use_bbn and hp.bbn_prior_weight > 0:
        try:
            cfg      = DATASET_CONFIG[dataset_key]
            sens_cols = [tk.replace("sens_","").replace("_train","")
                         for tk, _ in cfg["sens_attrs"]
                         if tk in data]
            bbn = BBNFairnessModel(target_col='label', sens_cols=sens_cols)
            bbn.fit(data['bbn_train_df'], y_train)
            if bbn.fitted:
                bbn_prior_train = bbn.predict_proba(data['bbn_train_df'])
                if verbose:
                    print(f"  BBN prior fitted. mean={bbn_prior_train.mean():.3f}")
        except Exception as e:
            if verbose:
                print(f"  BBN skipped: {e}")
    bbn_t = _tensor(bbn_prior_train)

    # ── Stage 1: encoder pre-training ─────────────────────────────────────────
    opt1 = optim.Adam(list(encoder.parameters()) + list(classifier.parameters()),
                      lr=hp.lr, weight_decay=1e-5)
    sched1 = optim.lr_scheduler.CosineAnnealingLR(opt1, T_max=hp.encoder_epochs, eta_min=1e-6)
    best_val_loss = np.inf
    patience_cnt  = 0
    best_enc_state = copy.deepcopy(encoder.state_dict())
    best_cls_state = copy.deepcopy(classifier.state_dict())

    for epoch in range(hp.encoder_epochs):
        encoder.train(); classifier.train()
        for batch in dl_tr:
            xb  = batch[0]
            yb  = batch[1]
            idx = dl_tr.dataset.tensors[0].shape  # just need batch indices
            z   = encoder(xb)
            logit = classifier(z)
            loss_cls = bce(logit, yb)

            if hp.bbn_prior_weight > 0 and hp.use_bbn:
                bi = torch.randint(0, len(bbn_t), (len(xb),))
                loss_bbn = F.binary_cross_entropy(
                    torch.sigmoid(logit), bbn_t[bi].clamp(0.05, 0.95))
                loss = loss_cls + hp.bbn_prior_weight * loss_bbn
            else:
                loss = loss_cls

            opt1.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(
                list(encoder.parameters()) + list(classifier.parameters()), 1.0)
            opt1.step()
        sched1.step()

        with torch.no_grad():
            encoder.eval(); classifier.eval()
            z_te   = encoder(X_te)
            val_loss = bce(classifier(z_te), _tensor(y_test)).item()
        if val_loss < best_val_loss - 1e-5:
            best_val_loss  = val_loss
            best_enc_state = copy.deepcopy(encoder.state_dict())
            best_cls_state = copy.deepcopy(classifier.state_dict())
            patience_cnt   = 0
        else:
            patience_cnt += 1
            if patience_cnt >= hp.early_patience:
                break

    encoder.load_state_dict(best_enc_state)
    classifier.load_state_dict(best_cls_state)

    proba_s1_tr = _get_proba(encoder, classifier, X_tr)
    proba_s1_te = _get_proba(encoder, classifier, X_te)
    acc_s1      = accuracy_score(y_test, (proba_s1_te >= 0.5).astype(int))
    if verbose:
        print(f"  Stage1 acc={acc_s1:.4f}")

    # ── Stage 2: adversarial debiasing ────────────────────────────────────────
    enc_lr = hp.lr * hp.encoder_lr_factor if hp.encoder_lr_factor > 0 else hp.lr * 0.1
    adv_lr = hp.lr * hp.adversary_lr_factor

    opt_enc = optim.Adam(list(encoder.parameters()) + list(classifier.parameters()),
                         lr=enc_lr, weight_decay=1e-5)
    opt_adv = optim.Adam(adversary.parameters(), lr=adv_lr, weight_decay=1e-5)

    best_s2_pred  = (proba_s1_te >= 0.5).astype(int)
    best_s2_score = np.inf
    acc_budget    = original_baseline_acc - hp.stage2_max_acc_drop

    for epoch in range(hp.fairness_epochs):
        lam = hp.lambda_adv_start + (hp.lambda_adv_max - hp.lambda_adv_start) * \
              min(1.0, epoch / max(1, hp.lambda_warmup_epochs))

        encoder.train(); classifier.train(); adversary.train()
        for batch in dl_tr:
            xb = batch[0]; yb = batch[1]
            sb_list = list(batch[2:])

            z       = encoder(xb)
            logit   = classifier(z)
            adv_out = adversary(z.detach())

            loss_adv = sum(bce_unw(ao, sb.to(DEVICE))
                           for ao, sb in zip(adv_out, sb_list)) / max(1, n_sens)
            opt_adv.zero_grad(); loss_adv.backward(); opt_adv.step()

            z       = encoder(xb)
            logit   = classifier(z)
            adv_out = adversary(z)

            loss_cls  = bce(logit, yb) * hp.cls_loss_weight_s2
            loss_conf = sum(bce_unw(ao, sb.to(DEVICE))
                            for ao, sb in zip(adv_out, sb_list)) / max(1, n_sens)
            loss_enc  = loss_cls - lam * loss_conf
            opt_enc.zero_grad(); loss_enc.backward(); nn.utils.clip_grad_norm_(
                list(encoder.parameters()) + list(classifier.parameters()), 1.0)
            opt_enc.step()

        if (epoch + 1) % 10 == 0 or epoch == hp.fairness_epochs - 1:
            proba_te = _get_proba(encoder, classifier, X_te)
            acc_now  = accuracy_score(y_test, (proba_te >= 0.5).astype(int))
            eo_now   = max((compute_eo(y_test, (proba_te >= 0.5).astype(int), s)
                            for s in s_test), default=0.0)
            dp_now   = max((compute_dp((proba_te >= 0.5).astype(int), s)
                            for s in s_test), default=0.0)
            penalty  = floor2(eo_now) * 20 + floor2(dp_now) * 10
            drop     = max(0.0, original_baseline_acc - acc_now)
            score    = penalty + (drop if penalty == 0.0 else 0.0)
            if score < best_s2_score and acc_now >= acc_budget:
                best_s2_score = score
                best_s2_pred  = (proba_te >= 0.5).astype(int)
                best_enc_state = copy.deepcopy(encoder.state_dict())
                best_cls_state = copy.deepcopy(classifier.state_dict())
            if verbose and (epoch + 1) % 50 == 0:
                print(f"  S2 ep{epoch+1}: acc={acc_now:.4f} EO={eo_now:.4f} "
                      f"DP={dp_now:.4f} lam={lam:.2f}")

    encoder.load_state_dict(best_enc_state)
    classifier.load_state_dict(best_cls_state)

    # ── Calibration + threshold sweep ─────────────────────────────────────────
    proba_tr_final = _get_proba(encoder, classifier, X_tr)
    proba_te_final = _get_proba(encoder, classifier, X_te)

    if hp.use_isotonic:
        proba_te_cal, proba_tr_cal = calibrate_proba(proba_tr_final, y_train, proba_te_final)
    else:
        proba_te_cal, proba_tr_cal = proba_te_final, proba_tr_final

    y_pred_final, tau_used = joint_threshold_sweep(
        y_test, proba_te_cal, s_test, hp, original_baseline_acc)

    acc_final = accuracy_score(y_test, y_pred_final)
    fair_final = compute_fairness_metrics(y_test, y_pred_final, s_dict)

    result = {"acc": acc_final, **fair_final}
    if verbose:
        drop = original_baseline_acc - acc_final
        eo_f = max(v for k, v in fair_final.items() if k.endswith("_eo"))
        dp_f = max(v for k, v in fair_final.items() if k.endswith("_dp"))
        print(f"  Final: acc={acc_final:.4f} drop={drop:+.4f} "
              f"EO_fl={floor2(eo_f):.2f} DP_fl={floor2(dp_f):.2f} tau={tau_used:.3f}")

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result


print("train_model defined.")
# %% ── CELL 8: Optuna HPO (fast) ─────────────────────────────────────────────
# Each trial uses SHORT epochs to find good hparams quickly.
# After HPO, one final full-length refit is done with the best hparams.
# Target: ~2-4 min per trial on CUDA, 50 trials ≈ 2-3 h per dataset.

# ── Epoch caps used ONLY during HPO search (not final refit) ─────────────────
_HPO_ENC_EPOCHS  = 60    # was up to 400 — kills trial time
_HPO_FAIR_EPOCHS = 80    # was up to 600 — kills trial time
_HPO_PATIENCE    = 10    # was 20-40
_HPO_BATCH_MIN   = 128   # force larger batches during search = fewer iters

def _hp_field_set(hp):
    return {f.name for f in dc_fields(hp)}

def _trial_to_hparams(trial, base: DatasetHParams,
                       fast: bool = True) -> DatasetHParams:
    valid = _hp_field_set(base)

    def _s(name, fn, *a, **kw):
        return fn(name, *a, **kw) if name in valid else getattr(base, name, None)

    hidden_dim  = _s("hidden_dim",           trial.suggest_categorical, [128, 192, 256])
    dropout     = _s("dropout",              trial.suggest_float, 0.05, 0.25, step=0.05)
    lr          = _s("lr",                   trial.suggest_float, 2e-4, 2e-3, log=True)
    batch_size  = _s("batch_size",           trial.suggest_categorical, [128, 256, 512])
    bbn_pw      = _s("bbn_prior_weight",     trial.suggest_float, 0.0, 0.30, step=0.05)
    enc_lr_f    = _s("encoder_lr_factor",    trial.suggest_float, 0.0, 0.8, step=0.1)
    adv_lr_f    = _s("adversary_lr_factor",  trial.suggest_float, 0.5, 2.0, step=0.25)
    lam_start   = _s("lambda_adv_start",     trial.suggest_float, 0.1, 0.8, step=0.1)
    lam_max     = _s("lambda_adv_max",       trial.suggest_float, 1.0, 8.0, step=0.5)
    lam_warm    = _s("lambda_warmup_epochs", trial.suggest_int,   5,  20,   step=5)
    cls_w_s2    = _s("cls_loss_weight_s2",   trial.suggest_float, 0.2, 0.8, step=0.1)
    fb_drop     = _s("acc_drop_fallback",    trial.suggest_float, 0.05, 0.10, step=0.01)
    iso         = _s("use_isotonic",         trial.suggest_categorical, [True, False])
    use_bbn     = _s("use_bbn",              trial.suggest_categorical, [True, False])
    use_gt      = _s("use_group_threshold",  trial.suggest_categorical, [True, False])
    gt_eps      = _s("group_thresh_eps",     trial.suggest_float, 0.1, 0.99, step=0.1)
    fine_w      = _s("fine_w",              trial.suggest_float, 0.1, 0.5, step=0.1)
    tau         = _s("tau",                  trial.suggest_float, 0.35, 0.65, step=0.05)

    if not use_bbn:
        bbn_pw = 0.0

    # During fast HPO trials, cap epochs hard regardless of what was suggested
    enc_epochs  = _HPO_ENC_EPOCHS  if fast else _s("encoder_epochs",  trial.suggest_int, 150, 400, step=25)
    fair_epochs = _HPO_FAIR_EPOCHS if fast else _s("fairness_epochs", trial.suggest_int, 150, 500, step=25)
    patience    = _HPO_PATIENCE    if fast else _s("early_patience",  trial.suggest_int, 15, 40, step=5)
    # Force large batch during search to reduce iters per epoch
    bs          = max(batch_size, _HPO_BATCH_MIN) if fast else batch_size

    kwargs = {f.name: getattr(base, f.name) for f in dc_fields(base)}
    overrides = dict(
        hidden_dim=hidden_dim, fairness_dim=hidden_dim // 2, dropout=dropout,
        lr=lr, batch_size=bs, encoder_epochs=enc_epochs, early_patience=patience,
        bbn_prior_weight=bbn_pw, fairness_epochs=fair_epochs,
        encoder_lr_factor=enc_lr_f, adversary_lr_factor=adv_lr_f,
        lambda_adv_start=lam_start, lambda_adv_max=lam_max,
        lambda_warmup_epochs=lam_warm, cls_loss_weight_s2=cls_w_s2,
        max_acc_drop=fb_drop, stage2_max_acc_drop=fb_drop + 0.01,
        acc_drop_fallback=fb_drop + 0.02,
        use_isotonic=iso, use_bbn=use_bbn,
        use_group_threshold=use_gt, group_thresh_eps=gt_eps,
        fine_w=fine_w, tau=tau,
    )
    for k, v in overrides.items():
        if k in valid and v is not None:
            kwargs[k] = v
    return DatasetHParams(**kwargs)


def _hpo_score(results: dict, baseline_acc: float) -> float:
    penalty = 0.0
    for k, v in results.items():
        if k.endswith("_eo") and not k.startswith("_"):
            penalty += 20.0 * floor2(v)
        if k.endswith("_dp") and not k.startswith("_"):
            penalty += 10.0 * floor2(v)
    drop = max(0.0, baseline_acc - results.get("acc", 0.0))
    return penalty + (drop if penalty == 0.0 else 0.0)


def _make_objective(data, dataset_key, baseline_acc, base_hp):
    def objective(trial):
        hp = _trial_to_hparams(trial, base_hp, fast=True)
        try:
            results = train_model(data, dataset_key, hp,
                                  original_baseline_acc=baseline_acc, verbose=False)
        except Exception as e:
            print(f"  [trial {trial.number}] ERROR: {e}")
            return 999.0
        finally:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        score   = _hpo_score(results, baseline_acc)
        eo_vals = [abs(v) for k, v in results.items()
                   if k.endswith("_eo") and not k.startswith("_")]
        dp_vals = [abs(v) for k, v in results.items()
                   if k.endswith("_dp") and not k.startswith("_")]
        max_eo  = max(eo_vals, default=1.0)
        max_dp  = max(dp_vals, default=1.0)
        drop    = baseline_acc - results.get("acc", 0.0)

        trial.set_user_attr("eo_floor",  floor2(max_eo))
        trial.set_user_attr("dp_floor",  floor2(max_dp))
        trial.set_user_attr("accuracy",  results.get("acc", 0.0))
        trial.set_user_attr("acc_drop",  drop)

        tag = "FAIR " if (floor2(max_eo) == 0.0 and floor2(max_dp) == 0.0) else "     "
        print(f"  {tag}[{trial.number:3d}] score={score:.4f} | "
              f"EO_fl={floor2(max_eo):.2f} DP_fl={floor2(max_dp):.2f} | "
              f"drop={drop:+.4f} | lam={hp.lambda_adv_max:.1f}")
        return score
    return objective


def _get_baseline_acc(data: dict) -> float:
    X_tr = to_dense(data['X_train_nn'])
    X_te = to_dense(data['X_test_nn'])
    mlp  = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=300,
                         random_state=SEED, early_stopping=True, verbose=False)
    mlp.fit(X_tr, data['y_train'])
    return float(accuracy_score(data['y_test'], mlp.predict(X_te)))


_PREPROCESS = {
    "adult":     preprocess_adult_for_fair_bbn,
    "compas":    preprocess_compas_for_fair_bbn,
    "german":    preprocess_german_for_fair_bbn,
    "bank":      preprocess_bank_for_fair_bbn,
    "lawschool": preprocess_lawschool_for_fair_bbn,
    "hospital":  preprocess_diabetes_hospital_for_fair_bbn,
}

# ── Full-length epoch schedule used for final refit ───────────────────────────
_REFIT_ENC_EPOCHS = {
    "adult": 250, "compas": 350, "german": 400,
    "bank": 100, "lawschool": 150, "hospital": 200,
}
_REFIT_FAIR_EPOCHS = {
    "adult": 400, "compas": 600, "german": 600,
    "bank": 150, "lawschool": 200, "hospital": 300,
}
_REFIT_PATIENCE = 25


def _refit_hp(hp: DatasetHParams, dataset_key: str) -> DatasetHParams:
    """Take HPO-found hparams and restore full epoch counts for final training."""
    valid = _hp_field_set(hp)
    kwargs = {f.name: getattr(hp, f.name) for f in dc_fields(hp)}
    if "encoder_epochs" in valid:
        kwargs["encoder_epochs"]  = _REFIT_ENC_EPOCHS.get(dataset_key, 250)
    if "fairness_epochs" in valid:
        kwargs["fairness_epochs"] = _REFIT_FAIR_EPOCHS.get(dataset_key, 400)
    if "early_patience" in valid:
        kwargs["early_patience"]  = _REFIT_PATIENCE
    # Use the actual suggested batch size (not the inflated HPO one)
    # batch_size was already stored in hp from trial suggestion
    return DatasetHParams(**kwargs)


def optimize_dataset(dataset_key: str, n_trials: int = 40,
                     timeout: int = 1800, n_refit_runs: int = 3):
    assert dataset_key in _PREPROCESS, f"Unknown dataset '{dataset_key}'"
    t0   = time.time()
    data = _PREPROCESS[dataset_key]()
    base = DEFAULT_HPARAMS[dataset_key]
    bacc = _get_baseline_acc(data)
    print(f"\n  {dataset_key.upper()}  baseline_acc={bacc:.4f}")
    print(f"  HPO: {n_trials} trials × ~2-4 min each  →  "
          f"~{n_trials*3//60}-{n_trials*4//60}h estimate")

    sampler = TPESampler(seed=SEED, n_startup_trials=max(5, n_trials // 5),
                         multivariate=True, warn_independent_sampling=False)
    study = optuna.create_study(
        direction="minimize", sampler=sampler,
        pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=5),
        study_name=f"hpo_{dataset_key}", load_if_exists=True)

    study.optimize(_make_objective(data, dataset_key, bacc, base),
                   n_trials=n_trials, timeout=timeout, n_jobs=1,
                   show_progress_bar=True, gc_after_trial=True)

    best    = study.best_trial
    fast_hp = _trial_to_hparams(best, base, fast=True)

    t_hpo = time.time() - t0
    print(f"\n  HPO done ({t_hpo/60:.1f}m) | trial#{best.number} "
          f"score={best.value:.4f} | "
          f"EO_fl={best.user_attrs.get('eo_floor','?')} "
          f"DP_fl={best.user_attrs.get('dp_floor','?')}")

    # ── Final refit with full epochs ──────────────────────────────────────────
    print(f"\n  Refitting with full epochs "
          f"(enc={_REFIT_ENC_EPOCHS.get(dataset_key,250)} "
          f"fair={_REFIT_FAIR_EPOCHS.get(dataset_key,400)}) "
          f"× {n_refit_runs} runs...")

    full_hp  = _refit_hp(fast_hp, dataset_key)
    refit_res = []
    for run in range(n_refit_runs):
        try:
            r = train_model(data, dataset_key, full_hp,
                            original_baseline_acc=bacc, verbose=True)
            refit_res.append(r)
            eo_v = max((abs(v) for k,v in r.items() if k.endswith("_eo") and not k.startswith("_")), default=1.0)
            dp_v = max((abs(v) for k,v in r.items() if k.endswith("_dp") and not k.startswith("_")), default=1.0)
            tag  = "ZERO" if (floor2(eo_v)==0.0 and floor2(dp_v)==0.0) else "    "
            print(f"  [{tag}] refit run {run}: acc={r.get('acc',0):.4f} "
                  f"EO_fl={floor2(eo_v):.2f} DP_fl={floor2(dp_v):.2f}")
        except Exception as e:
            print(f"  refit run {run} error: {e}")
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    DEFAULT_HPARAMS[dataset_key] = full_hp

    path = os.path.join(RESULTS_DIR, f"best_hparams_{dataset_key}.json")
    hp_dict = {f.name: getattr(full_hp, f.name) for f in dc_fields(full_hp)}
    hp_dict["_meta"] = {
        "dataset": dataset_key, "baseline_acc": bacc,
        "best_hpo_score": best.value, "elapsed_min": round((time.time()-t0)/60, 2),
        "n_trials": n_trials, "eo_floor": best.user_attrs.get("eo_floor"),
        "dp_floor": best.user_attrs.get("dp_floor"),
    }
    with open(path, "w") as f:
        json.dump(hp_dict, f, indent=2, default=str)

    return full_hp, data, bacc, refit_res


def tune_all_datasets(
    datasets=None,
    n_trials_per_dataset: int = 30,
    timeout_per_dataset:  int = 3600,
    n_refit_runs:         int = 3,
    continue_on_error:    bool = True,
):
    all_valid = list(_PREPROCESS.keys())
    if datasets is None:
        datasets = all_valid

    est_min = n_trials_per_dataset * 3   # ~3 min/trial on CUDA
    print("="*65)
    print("  ZERO-FAIRNESS HPO  (fast-trial + full refit)")
    print(f"  Target: floor(EO)=0.00 AND floor(DP)=0.00, acc drop < 5%")
    print(f"  Device: {DEVICE}")
    print(f"  Datasets      : {datasets}")
    print(f"  Trials/dataset: {n_trials_per_dataset}  "
          f"(~{est_min}m/dataset, ~{est_min*len(datasets)//60}h total)")
    print(f"  Timeout/dataset: {timeout_per_dataset//60}m")
    print("="*65)

    results_cache = {}
    failed = []
    t0 = time.time()

    for i, ds in enumerate(datasets, 1):
        print(f"\n{'─'*65}")
        print(f"  [{i}/{len(datasets)}] {ds.upper()}")
        print(f"{'─'*65}")
        try:
            full_hp, data, bacc, refit_res = optimize_dataset(
                ds, n_trials_per_dataset, timeout_per_dataset, n_refit_runs)
            results_cache[ds] = (full_hp, data, bacc, refit_res)
        except Exception as e:
            print(f"  FAILED {ds}: {e}")
            failed.append(ds)
            if not continue_on_error:
                raise

    print(f"\n  Total time: {(time.time()-t0)/60:.1f}m")
    print(f"  Done: {[d for d in datasets if d not in failed]}")
    print(f"  Failed: {failed}")

    _print_final_table(results_cache)

    all_path = os.path.join(RESULTS_DIR, "best_hparams_all.json")
    with open(all_path, "w") as f:
        json.dump({ds: {fi.name: getattr(hp, fi.name) for fi in dc_fields(hp)}
                   for ds, (hp, _, _, _) in results_cache.items()},
                  f, indent=2, default=str)
    print(f"\n  Saved → {all_path}")
    return results_cache


def _print_final_table(results_cache: dict):
    print(f"\n{'='*65}")
    print(f"  FINAL RESULTS")
    print(f"  {'Dataset':12} {'Base':>7} {'Ours':>7} {'Drop':>7} "
          f"{'EO_fl':>6} {'DP_fl':>6}  Status")
    print("  " + "─"*55)

    for ds, (hp, data, bacc, refit_res) in results_cache.items():
        if not refit_res:
            print(f"  {ds.upper():12} no results")
            continue
        keys = refit_res[0].keys()
        avg  = {k: float(np.mean([r[k] for r in refit_res if k in r]))
                for k in keys if isinstance(refit_res[0].get(k), (int, float, np.floating))}
        eo_vals = [abs(avg[k]) for k in avg if k.endswith("_eo") and not k.startswith("_")]
        dp_vals = [abs(avg[k]) for k in avg if k.endswith("_dp") and not k.startswith("_")]
        max_eo  = max(eo_vals, default=1.0)
        max_dp  = max(dp_vals, default=1.0)
        drop    = bacc - avg.get("acc", 0.0)
        ef, df  = floor2(max_eo), floor2(max_dp)
        tag = "ZERO " if (ef==0.0 and df==0.0) else \
              "near " if (ef<=0.01 and df<=0.01) else "FAIL "
        print(f"  {ds.upper():12} {bacc:7.4f} {avg.get('acc',0):7.4f} "
              f"{drop:+7.4f} {ef:6.2f} {df:6.2f}  {tag}")

    out = os.path.join(RESULTS_DIR, "final_results.json")
    with open(out, "w") as f:
        json.dump({ds: {
            "baseline_acc": bacc,
            "our_acc": float(np.mean([r.get("acc",0) for r in rr])) if rr else 0,
            "refit_runs": len(rr),
        } for ds, (_, _, bacc, rr) in results_cache.items()},
        f, indent=2)
    print(f"\n  Saved → {out}")


print("HPO functions defined.")
# %% ── CELL 9: RUN ────────────────────────────────────────────────────────────
# Time estimates (CUDA T4):
#   ~2-3 min/trial during HPO (short epochs)
#   ~15-20 min/dataset for final refit (3 runs × full epochs)
#
#   30 trials × 2.5 min + 20 min refit = ~1.5 h/dataset
#   6 datasets total ≈ 8-10 h for full run

# ── Smoke test (~30 min) ───────────────────────────────────────────────────────
# results_cache = tune_all_datasets(
#     datasets=['adult'], n_trials_per_dataset=5,
#     timeout_per_dataset=600, n_refit_runs=1,
# )

# ── 2 datasets (~3 h) ─────────────────────────────────────────────────────────
# results_cache = tune_all_datasets(
#     datasets=['adult', 'compas'], n_trials_per_dataset=25,
#     timeout_per_dataset=3600, n_refit_runs=3,
# )

# ── 3 datasets (~5 h) ─────────────────────────────────────────────────────────
# results_cache = tune_all_datasets(
#     datasets=['adult', 'compas', 'german'], n_trials_per_dataset=30,
#     timeout_per_dataset=3600, n_refit_runs=3,
# )

# ── All 6 datasets (~9-10 h) ──────────────────────────────────────────────────
results_cache = tune_all_datasets(
    datasets=['adult', 'compas', 'german', 'bank', 'lawschool', 'hospital'],
    n_trials_per_dataset=30,
    timeout_per_dataset=3600,
    n_refit_runs=3,
    continue_on_error=True,
)

print("\nDone.")

/usr/local/lib/python3.12/dist-packages/pgmpy/estimators/__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in a future release. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


Device: cuda  |  CUDA: True
Optuna: True
DatasetHParams and DEFAULT_HPARAMS defined.
Datasets: ['adult', 'compas', 'german', 'bank', 'lawschool', 'hospital']
Utility helpers and fairness metrics defined.
All 6 preprocessing functions defined.
BBN model and neural network modules defined.
Threshold sweep defined.
train_model defined.
HPO functions defined.
  ZERO-FAIRNESS HPO  (fast-trial + full refit)
  Target: floor(EO)=0.00 AND floor(DP)=0.00, acc drop < 5%
  Device: cuda
  Datasets      : ['adult', 'compas', 'german', 'bank', 'lawschool', 'hospital']
  Trials/dataset: 30  (~90m/dataset, ~9h total)
  Timeout/dataset: 60m

─────────────────────────────────────────────────────────────────
  [1/6] ADULT
─────────────────────────────────────────────────────────────────

  ADULT  baseline_acc=0.8618
  HPO: 30 trials × ~2-4 min each  →  ~1-2h estimate


  0%|          | 0/30 [00:00<?, ?it/s]

       [  0] score=3.4000 | EO_fl=0.05 DP_fl=0.15 | drop=+0.0240 | lam=7.0
       [  1] score=2.3000 | EO_fl=0.03 DP_fl=0.10 | drop=+0.0167 | lam=1.5
       [  2] score=2.4000 | EO_fl=0.02 DP_fl=0.12 | drop=+0.0948 | lam=7.0
       [  3] score=1.8000 | EO_fl=0.02 DP_fl=0.10 | drop=+0.0157 | lam=5.5
       [  4] score=3.1000 | EO_fl=0.09 DP_fl=0.02 | drop=+0.0597 | lam=7.5
       [  5] score=4.5000 | EO_fl=0.06 DP_fl=0.16 | drop=+0.0506 | lam=6.0
       [  6] score=4.3000 | EO_fl=0.06 DP_fl=0.16 | drop=+0.0307 | lam=7.5
       [  7] score=4.4000 | EO_fl=0.06 DP_fl=0.17 | drop=+0.0688 | lam=2.5
       [  8] score=3.5000 | EO_fl=0.07 DP_fl=0.12 | drop=+0.0897 | lam=5.0
       [  9] score=3.9000 | EO_fl=0.05 DP_fl=0.15 | drop=+0.0522 | lam=2.5
       [ 10] score=4.0000 | EO_fl=0.10 DP_fl=0.11 | drop=+0.0554 | lam=4.0
       [ 11] score=3.5000 | EO_fl=0.06 DP_fl=0.14 | drop=+0.0111 | lam=3.0
       [ 12] score=2.0000 | EO_fl=0.02 DP_fl=0.09 | drop=+0.0317 | lam=2.5
       [ 13] score=1.8000

  0%|          | 0/30 [00:00<?, ?it/s]

...

       [  0] score=0.2000 | EO_fl=0.00 DP_fl=0.02 | drop=+0.0217 | lam=7.0
  [trial 1] ERROR: Expected more than 1 value per channel when training, got input size torch.Size([1, 128])
  [trial 2] ERROR: Expected more than 1 value per channel when training, got input size torch.Size([1, 128])
  [trial 3] ERROR: Expected more than 1 value per channel when training, got input size torch.Size([1, 192])
       [  4] score=0.5000 | EO_fl=0.01 DP_fl=0.01 | drop=+0.0809 | lam=7.5
  [trial 5] ERROR: Expected more than 1 value per channel when training, got input size torch.Size([1, 256])
  FAIR [  6] score=0.0019 | EO_fl=0.00 DP_fl=0.00 | drop=+0.0019 | lam=7.5
       [  7] score=0.4000 | EO_fl=0.02 DP_fl=0.00 | drop=+0.0491 | lam=8.0
       [  8] score=0.2000 | EO_fl=0.00 DP_fl=0.01 | drop=+0.0504 | lam=7.5
       [  9] score=0.1000 | EO_fl=0.00 DP_fl=0.01 | drop=-0.0025 | lam=8.0
       [ 10] score=0.2000 | EO_fl=0.00 DP_fl=0.02 | drop=+0.0089 | lam=8.0
       [ 11] score=0.4000 | EO_fl=0.01 

  0%|          | 0/30 [00:00<?, ?it/s]

  FAIR [  0] score=0.0011 | EO_fl=0.00 DP_fl=0.00 | drop=+0.0011 | lam=7.0
  FAIR [  1] score=0.0016 | EO_fl=0.00 DP_fl=0.00 | drop=+0.0016 | lam=1.5
       [  2] score=1.6000 | EO_fl=0.03 DP_fl=0.04 | drop=+0.0100 | lam=7.0
       [  3] score=7.6000 | EO_fl=0.26 DP_fl=0.10 | drop=+0.0022 | lam=5.5
  FAIR [  4] score=0.0020 | EO_fl=0.00 DP_fl=0.00 | drop=+0.0020 | lam=7.5
       [  5] score=2.2000 | EO_fl=0.10 DP_fl=0.02 | drop=+0.0004 | lam=6.0
  FAIR [  6] score=0.0018 | EO_fl=0.00 DP_fl=0.00 | drop=+0.0018 | lam=7.5
  FAIR [  7] score=0.0018 | EO_fl=0.00 DP_fl=0.00 | drop=+0.0018 | lam=6.0
  FAIR [  8] score=0.0016 | EO_fl=0.00 DP_fl=0.00 | drop=+0.0016 | lam=6.5
       [  9] score=7.2000 | EO_fl=0.26 DP_fl=0.10 | drop=+0.0027 | lam=3.0
  FAIR [ 10] score=0.0018 | EO_fl=0.00 DP_fl=0.00 | drop=+0.0018 | lam=5.0
  FAIR [ 11] score=0.0016 | EO_fl=0.00 DP_fl=0.00 | drop=+0.0016 | lam=1.5
  FAIR [ 12] score=0.0018 | EO_fl=0.00 DP_fl=0.00 | drop=+0.0018 | lam=1.5
  FAIR [ 13] score=0.0016

  0%|          | 0/30 [00:00<?, ?it/s]

  FAIR [  0] score=0.0627 | EO_fl=0.00 DP_fl=0.00 | drop=+0.0627 | lam=7.0
  FAIR [  1] score=0.0544 | EO_fl=0.00 DP_fl=0.00 | drop=+0.0544 | lam=1.5
  FAIR [  2] score=0.0515 | EO_fl=0.00 DP_fl=0.00 | drop=+0.0515 | lam=7.0
  FAIR [  3] score=0.0557 | EO_fl=0.00 DP_fl=0.00 | drop=+0.0557 | lam=5.5
  FAIR [  4] score=0.0567 | EO_fl=0.00 DP_fl=0.00 | drop=+0.0567 | lam=7.5
  FAIR [  5] score=0.0442 | EO_fl=0.00 DP_fl=0.00 | drop=+0.0442 | lam=6.0
  FAIR [  6] score=0.0675 | EO_fl=0.00 DP_fl=0.00 | drop=+0.0675 | lam=8.0
       [  7] score=0.2000 | EO_fl=0.01 DP_fl=0.00 | drop=+0.0413 | lam=5.0
       [  8] score=0.2000 | EO_fl=0.01 DP_fl=0.00 | drop=+0.0250 | lam=3.0
  FAIR [  9] score=0.0683 | EO_fl=0.00 DP_fl=0.00 | drop=+0.0683 | lam=8.0
       [ 10] score=0.2000 | EO_fl=0.01 DP_fl=0.00 | drop=+0.0345 | lam=7.5
  FAIR [ 11] score=0.0245 | EO_fl=0.00 DP_fl=0.00 | drop=+0.0245 | lam=7.0
  FAIR [ 12] score=0.0669 | EO_fl=0.00 DP_fl=0.00 | drop=+0.0669 | lam=3.5
  FAIR [ 13] score=0.0601